<a href="https://colab.research.google.com/github/kristen531/Fall-Detection-Models/blob/main/PredictionBasedOnCholestroCgm_levehHb1acl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [244]:
import pandas as pd

# 读取 CSV 文件
nodes_df = pd.read_csv("nodes.csv")
edges_df = pd.read_csv("edges.csv")
health_df = pd.read_csv("health_data.csv")

# 清理字段名（去掉 p.、h. 前缀）
nodes_df.columns = nodes_df.columns.str.replace("p.", "", regex=False)
health_df.columns = health_df.columns.str.replace("h.", "", regex=False)

# 显示确认
print("✅ 节点数据：", nodes_df.columns.tolist())
print("✅ 健康数据：", health_df.columns.tolist())
print("✅ 边数据：", edges_df.columns.tolist())


✅ 节点数据： ['node_id', 'patient_id', 'health_risk', 'gender', 'created_at']
✅ 健康数据： ['p.patient_id', 'cgm_level', 'blood_pressure', 'heart_rate', 'cholesterol', 'insulin_intake', 'food_intake', 'activity_level', 'weight', 'hb1ac']
✅ 边数据： ['source', 'target', 'relation']


In [245]:
# 1. 改名 health_df 的 patient_id，以便匹配 nodes_df
health_df = health_df.rename(columns={"p.patient_id": "patient_id"})

# 2. 合并两个 DataFrame：用 patient_id 作为 key

merged_df = pd.merge(nodes_df, health_df, on="patient_id", how="inner")

# ⬇️ 加这里
def define_risk(row):
    if row["hb1ac"] > 8.0 and row["cgm_level"] > 141 or row["cholesterol"] > 220:
        return "high"
    elif row["hb1ac"] > 6.5 and row["cgm_level"] > 71 or row["cholesterol"] > 180:
        return "medium"
    else:
        return "low"

merged_df["health_risk"] = merged_df.apply(define_risk, axis=1)


# 3. 显示合并后的结果
print("✅ 合并后字段：", merged_df.columns.tolist())
print("📊 合并后的样本：")
print(merged_df.head())


✅ 合并后字段： ['node_id', 'patient_id', 'health_risk', 'gender', 'created_at', 'cgm_level', 'blood_pressure', 'heart_rate', 'cholesterol', 'insulin_intake', 'food_intake', 'activity_level', 'weight', 'hb1ac']
📊 合并后的样本：
   node_id patient_id health_risk  gender       created_at  cgm_level  \
0       46      P1000        high    Male   1/19/2023 8:00      114.3   
1       47      P1001      medium  Female   1/19/2023 8:30      165.2   
2       48      P1002        high  Female  1/19/2023 10:00      170.9   
3       49      P1003      medium    Male   1/20/2023 8:30      130.4   
4       50      P1004      medium  Female  1/20/2023 11:30      117.9   

  blood_pressure  heart_rate  cholesterol  insulin_intake       food_intake  \
0         130/85          76        246.1               1  Oatmeal and milk   
1         118/75          73        165.7              10    Salmon, quinoa   
2         130/85          82        239.7               1     Rice, chicken   
3         135/88          72   

In [246]:
import torch

# 🔹 拆解 blood_pressure 为 systolic 和 diastolic
bp_split = merged_df["blood_pressure"].str.split("/", expand=True)
merged_df["systolic"] = pd.to_numeric(bp_split[0], errors="coerce")
merged_df["diastolic"] = pd.to_numeric(bp_split[1], errors="coerce")

# 🟡 我们只用这两个特征
x_features = ["cgm_level", "hb1ac", "cholesterol"]


# 提取特征值并转换成 tensor
X = merged_df[x_features].apply(pd.to_numeric, errors="coerce").fillna(0)
x = torch.tensor(X.values, dtype=torch.float)

print("✅ 使用的特征 shape:", x.shape)

# 🔹 标签编码
y = merged_df["health_risk"].astype("category").cat.codes
y = torch.tensor(y.values, dtype=torch.long)

print("✅ x shape:", x.shape)
print("✅ y shape:", y.shape)
print("🧠 类别编码顺序：", merged_df["health_risk"].astype("category").cat.categories.tolist())


✅ 使用的特征 shape: torch.Size([26, 3])
✅ x shape: torch.Size([26, 3])
✅ y shape: torch.Size([26])
🧠 类别编码顺序： ['high', 'low', 'medium']


In [247]:
from torch_geometric.data import Data

# 过滤无效边（确保 source 和 target 都在有效 node_id 中）
valid_node_ids = set(merged_df["node_id"])
edges_df = edges_df[
    edges_df["source"].isin(valid_node_ids) & edges_df["target"].isin(valid_node_ids)
].reset_index(drop=True)

# 重新映射 node_id 为 0 ~ N-1（PyG 要求的格式）
node_id_map = {nid: i for i, nid in enumerate(merged_df["node_id"])}
edges_df["source"] = edges_df["source"].map(node_id_map)
edges_df["target"] = edges_df["target"].map(node_id_map)

# 转换为 PyTorch Tensor
edge_index = torch.tensor(edges_df[["source", "target"]].values.T, dtype=torch.long)

# 创建 PyG 的 Data 对象
data = Data(x=x, edge_index=edge_index, y=y)

print("✅ 图数据已准备好：")
print(data)


✅ 图数据已准备好：
Data(x=[26, 3], edge_index=[2, 0], y=[26])


In [248]:
import torch.nn as nn
from torch_geometric.nn import GATConv

class GATModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GATModel, self).__init__()
        self.gat1 = GATConv(input_dim, hidden_dim, heads=4, dropout=0.6)
        self.gat2 = GATConv(hidden_dim * 4, output_dim, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = torch.relu(x)
        x = self.gat2(x, edge_index)
        return x


In [249]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GATModel(input_dim=3, hidden_dim=8, output_dim=3).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()


In [250]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

# 训练 200 次
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")


Epoch 020, Loss: 5.2806
Epoch 040, Loss: 2.3036
Epoch 060, Loss: 2.4614
Epoch 080, Loss: 2.7639
Epoch 100, Loss: 1.1175
Epoch 120, Loss: 2.9263
Epoch 140, Loss: 2.8233
Epoch 160, Loss: 0.9094
Epoch 180, Loss: 1.2251
Epoch 200, Loss: 1.0102


In [251]:
model.eval()  # 切换到推理模式
out = model(data.x, data.edge_index)
preds = out.argmax(dim=1)  # 拿到每个节点预测的标签编号

In [252]:
# 取得类别顺序（用于将预测编号转为文字标签）
label_mapping = merged_df["health_risk"].astype("category").cat.categories.tolist()
pred_labels = [label_mapping[p] for p in preds.tolist()]

In [253]:
!pip install neo4j --quiet

In [254]:
from neo4j import GraphDatabase

uri = "neo4j+s://286a1468.databases.neo4j.io"
username = "neo4j"  # 默认是 neo4j，或你设置的用户名
password = "JoZo-BELo0w7KEg9lVseaXMqx4XfEQx-wfnXu2soMpM"  # 你的密码

driver = GraphDatabase.driver(uri, auth=(username, password))


In [255]:
# 确保 node_id 和预测标签一样长
assert len(merged_df["node_id"]) == len(pred_labels)

with driver.session() as session:
    for node_id, pred in zip(merged_df["node_id"], pred_labels):
        session.run("""
            MATCH (n:Patient)
            WHERE id(n) = $id
            SET n.prediction = $pred
        """, id=int(node_id), pred=pred)

print("✅ 预测结果已成功写入 Neo4j!")


✅ 预测结果已成功写入 Neo4j!
